На основе представленного датасета будем решать задачу классификации. Попытаемся прогнозировать конверсию, т.е. потратит пользователь деньги на цифровые услуги или нет. Целевая колонка Conversion бинарная, соответственно и классификация бинарная.

Представленные в датасете данные не имеют пропусков, что нормально для маркетинговой аналитики на цифровых платформах. Будем смотреть на выбросы,корреляцию данных, вариативность и пробовать разные модели классификации.


In [5]:
import phik as phik
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### Загрузка данных


In [32]:
df = pd.read_csv("./digital_marketing_campaign_dataset.csv", sep=",")

### Основная информация о данных


In [ ]:
print("df shape =", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Может ли колонка CustomerID быть индексом


In [35]:
df = df.set_index("CustomerID", inplace=False)

### Работа с цифровыми колонками

-   Отделить цифровые колонки
-   Посмотреть и оценить корреляции (очень желательно не одним исследованием)
-   Провести оценку выбросов
-   Оценить изменчивость относительно таргета

Самостоятельный поиск способов дополнительной оценки цифровых признаков


In [ ]:
df_numerics = df[df._get_numeric_data().columns]
df_numerics.head()

In [12]:
# pd.plotting.scatter_matrix(
#     df_numerics, figsize=(15, 15), diagonal="kde"
# )
# plt.show()

In [ ]:
df[["CustomerID", "Conversion"]]

In [ ]:
plt.scatter(df["CustomerID"], df["Conversion"])

In [ ]:
plt.scatter(
    df[df["Conversion"] == 1]["PreviousPurchases"],
    df[df["Conversion"] == 1]["LoyaltyPoints"],
    color="orange",
    label="Pay",
)
plt.scatter(
    df[df["Conversion"] == 0]["PreviousPurchases"],
    df[df["Conversion"] == 0]["LoyaltyPoints"],
    color="blue",
    label="Don't pay",
)

In [ ]:
df_numerics.hist(color="k", bins=30, figsize=(15, 10))
plt.show()

In [ ]:
sns.heatmap(df_numerics.corr(method='kendall'), annot=True, fmt='.2f')

### Работа с категориальными признаками

-   Отделить категориальные колонки
-   Оценить изменчивость относительно таргета

Самостоятельный поиск способов дополнительной оценки цифровых признаков


In [ ]:
categorial_cols = list(set(df.columns) - set(df._get_numeric_data().columns))
print(categorial_cols)

In [ ]:
for col in df[categorial_cols].columns:
    print(df[col].value_counts())
    print("\n")

In [ ]:
df.drop(['AdvertisingPlatform', 'AdvertisingTool'], axis=1, inplace=True)
categorial_cols = list(set(df.columns) - set(df._get_numeric_data().columns))
print(categorial_cols)

In [ ]:
fig, ax = plt.subplots(len(categorial_cols), 1)
for i in range(len(categorial_cols)):
    sns.countplot(x=categorial_cols[i], hue="Conversion", data=df, ax=ax[i])

In [ ]:
pd.crosstab(df["CampaignChannel"], df["CampaignType"])

#### Кодируем оставшиеся категориальные признаки one-hot-encoding


In [ ]:
df.head()

In [39]:
for category in categorial_cols:
    df = pd.concat([df, pd.get_dummies(df[category], prefix=category)],
                   axis=1,
                   )
    df.drop([category],
            axis=1,
            inplace=True)

In [ ]:
df.head()

In [51]:
df = df*1

### Разделение данных на признаки и таргет

Оценка баланса классов


In [59]:
from sklearn.model_selection import train_test_split

X = df.drop('Conversion', axis=1)
Y = df['Conversion']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25)

### Построение модели классификатора на базе логистической регрессии

-   Использование конвейера обязательно
-   Подобрать оптимальные параметры модели
-   Нарисовать Confusion Matrix
-   Получить оценки Precision, Recall
-   Построить PR кривую
-   Посчитать AUC


In [60]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [ ]:
pipe_log = Pipeline([('Scaler', StandardScaler()),
                    ('LR', LogisticRegression(class_weight='balanced'))])
pipe_log.fit(X_train, Y_train)

print(
    f'Accuracy равно: {accuracy_score(Y_test, pipe_log.predict(X_test)):.3f}')
print(
    f'Precision равно: {precision_score(Y_test, pipe_log.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe_log.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe_log.predict(X_test)):.3f}')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import auc
from sklearn.metrics import PrecisionRecallDisplay

precision_l, recall_l, thresholds_l = precision_recall_curve(
    Y_test, pipe_log.predict_proba(X_test)[:, 1])
# precision_m, recall_m, thresholds_m = precision_recall_curve(
#     Y_test, pipe_mm.predict_proba(X_test)[:, 1])

PrecisionRecallDisplay(precision=precision_l, recall=recall_l).plot()
# PrecisionRecallDisplay(precision=precision_m, recall=recall_m).plot()

In [ ]:
# Посчитаем PR-AUC
auc(recall_l, precision_l)

### Построение модели KNN классификатора

-   Использование конвейера обязательно
-   Подобрать оптимальные параметры модели
-   Нарисовать Confusion Matrix
-   Получить оценки Precision, Recall
-   Построить PR кривую
-   Посчитать AUC


### Обязательно оценить все модели не только на F1-Score, но и на AUC. Выбрать модель. Обосновать выбор. Провести дополнительное тестирование модели на тех же данных с использованием cross-validation
